<div style="background:#03045E;padding:28px 32px;border-radius:16px;font-family:Arial,sans-serif;">
<img src="../assets/jekacode-logo.png" alt="Jekacode" width="240"/>
<p style="color:#16D365;font-size:12px;letter-spacing:2.5px;margin:18px 0 6px 0;">JEKACODE AI ENGINEERING · WEEK 7 · DAY 2</p>
<h1 style="color:#ffffff;margin:0;font-size:28px;">Lab: research assistant</h1>
<p style="color:#d7deea;margin:10px 0 0 0;font-size:16px;">Run twice. Did structure hold?</p>
</div>


## What we want to achieve

Streamlit research_assistant.py and/or Gradio. Document tool use. Watch inconsistency.

## Tools you need (names only)

Same keys as Day 1.

## How to (do these before the first code cell if you have not)

```bash
streamlit run week07-agents/research_assistant.py
python week07-agents/gradio_app.py
```
Read every comment at the top of `research_assistant.py`.

## What goes on behind the scenes

If run 2 picks a weaker plan, that is **inconsistency**. You still ship: add a fixed outline in the system prompt. Behind the scenes: `outline()` is one `ask()`, `search_handbook()` is Python, the report is a second `ask()`. Two latencies.

**Keys & installs (bookmark):** [../guides/HOW_TO.md](../guides/HOW_TO.md) · **Terms:** [../guides/AI_ENGINEERING_TERMS.md](../guides/AI_ENGINEERING_TERMS.md)


In [ ]:
# --- Why this cell exists (read once) ---
# Python only finds packages that live on a list of folders called sys.path.
# This notebook sits in a week folder. The jekacode helper lives one folder up.
# Novices: you are not "hacking". You are telling Python where the course lives.

import sys
# sys = the "system" module. We use it to change where Python looks for imports.

from pathlib import Path
# Path is a friendly way to talk about folders. It works on Mac, Windows, and Linux.

root = Path.cwd()
# cwd = current working directory = "the folder this notebook thinks it is in".

if not (root / "jekacode").exists():
    # If we cannot see the jekacode folder here, we are inside week01, week02, ...
    root = root.parent
    # parent = the folder above this one (the course root).

if str(root) not in sys.path:
    sys.path.append(str(root))
    # Now `from jekacode.ai import ask` can succeed.

print("Course folder Python will use:", root)
print("You should see jekacode inside that folder.")


## Theory: why agents fail in public

- The model **skips** the tool and invents a citation → **hallucination**.
- The model calls the tool with a useless query → garbage in.
- Two runs, two outlines → **inconsistency**. Fix: numbered headings in the system prompt.

**User story:** *As a parent I want a short briefing that admits what we do not know.*

Read `research_assistant.py`. Then recreate the loop here:


In [ ]:
from pathlib import Path
from jekacode.ai import ask

# Load a local textbook (the Jekacode handbook). This is the tool's memory.
handbook = Path("../knowledge/jekacode-handbook.md").read_text()
topic = "Explain the Jekacode AI Engineering programme to a parent"

def search_handbook(query: str) -> str:
    # Cheap retrieval: share words with the query (Week 6 idea inside an agent).
    words = set(query.lower().split())
    parts = [p.strip() for p in handbook.split("##") if p.strip()]
    parts = sorted(parts, key=lambda p: sum(1 for w in words if w in p.lower()), reverse=True)
    return (parts[0][:800] if parts else "Nothing found.")

notes = search_handbook(topic)
print("TOOL\n", notes[:400], "\n")
print(ask(
    f"Headings: Problem, Facts we have, What we still don't know, Next step.\nTopic: {topic}\nNotes:\n{notes}",
    system="Honest intern. If notes are weak, say so.",
    provider="gemini",
))
